# Team Lannisters: BioNER demo backend
Run this **interactively** (not Save & Run All) and keep the tab open during the demo.

Inputs (right panel -> **Add Input**):
1. your `bioner-code` dataset (latest version)
2. the **output of your finished `kaggle_train` run** (Add Input -> Your Work / Notebooks -> pick it). This supplies the trained checkpoints and `submission.csv`.

Settings: **Accelerator = GPU T4** (fast answers; CPU also works, just slower), **Internet = On**.

In [ ]:
import glob, os, shutil
W = '/kaggle/working/bioner'
# the bundle is the only input with requirements.txt next to src/ (a training output has a bioner/src copy without it)
cfg = [c for c in glob.glob('/kaggle/input/**/src/config.py', recursive=True)
       if os.path.exists(os.path.join(os.path.dirname(os.path.dirname(c)), 'requirements.txt'))]
assert cfg, 'bioner-code dataset not found: Add Input -> your dataset'
root = os.path.dirname(os.path.dirname(cfg[0]))
for sub in ['src', 'data', 'backend']:                 # refresh code, keep existing checkpoints
    shutil.rmtree(f'{W}/{sub}', ignore_errors=True)
    shutil.copytree(f'{root}/{sub}', f'{W}/{sub}')
ck = glob.glob('/kaggle/input/**/bioner/checkpoints/span/span_model.pt', recursive=True)
if ck and not os.path.exists(f'{W}/checkpoints/span'):   # reuse a finished training run
    prev = ck[0].split('/checkpoints/')[0]
    for sub in ['checkpoints', 'submission']:
        if os.path.exists(f'{prev}/{sub}'):
            shutil.copytree(f'{prev}/{sub}', f'{W}/{sub}', dirs_exist_ok=True)
    print('copied checkpoints from', prev)
%cd /kaggle/working/bioner
!pip install -q seqeval fastapi uvicorn   # torch, transformers, pandas, sklearn are preinstalled
assert _exit_code == 0, 'command failed, see output above'
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'no GPU: running on CPU'
assert _exit_code == 0, 'command failed, see output above'

## Start the FastAPI backend in the background

In [ ]:
import subprocess
server = subprocess.Popen('python -m uvicorn backend.app:app --host 0.0.0.0 --port 8000',
                          shell=True, stdout=open('backend.log', 'w'), stderr=subprocess.STDOUT)

In [ ]:
import json, time, urllib.request
def call(path, body=None):
    req = urllib.request.Request('http://localhost:8000' + path,
                                 data=json.dumps(body).encode() if body else None,
                                 headers={'Content-Type': 'application/json'})
    return json.loads(urllib.request.urlopen(req).read())
up = False
for _ in range(60):                     # loading two BioBERTs takes ~30-60 s
    if server.poll() is not None:        # the server process died: show why at once
        break
    try:
        print(call('/health')); up = True; break
    except Exception:
        time.sleep(5)
if not up:
    print(open('backend.log').read()[-3000:]); raise RuntimeError('backend did not start')

## Phase 8 checks: every endpoint

In [ ]:
text = 'Mutations in the BRCA1 gene cause hereditary breast and ovarian cancer.'
s = call('/predict', {'text': text, 'model': 'span'})
print('tokens+tags:', list(zip(s['tokens'], s['tags'])))
print('span entities:', s['entities'])
print('top 5 candidates:', s['candidates'][:5])
print('grid:', len(s['grid']), 'x', len(s['grid'][0]), ' threshold:', s['threshold'])
b = call('/predict', {'text': text, 'model': 'baseline'})
print('baseline entities:', b['entities'])
c = call('/compare', {'text': text})
print('disagree at word indices:', c['disagree'])
print('examples:', call('/examples')['examples'][:3])

## Open a public tunnel so the frontend on your laptop can reach this backend
Cloudflare quick tunnel: no account needed. It only forwards HTTP to our own server; all predictions are still computed by our models here on Kaggle.

In [ ]:
![ -f cloudflared ] || wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
assert _exit_code == 0, 'command failed, see output above'
!chmod +x cloudflared
assert _exit_code == 0, 'command failed, see output above'
tunnel = subprocess.Popen('./cloudflared tunnel --url http://localhost:8000', shell=True,
                          stdout=open('tunnel.log', 'w'), stderr=subprocess.STDOUT)

In [ ]:
import re
for _ in range(30):
    urls = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
    if urls:
        print('BACKEND URL (paste into the frontend):', urls[0]); break
    time.sleep(2)
else:
    print(open('tunnel.log').read()[-2000:])